# Step 4A.10 — B3: Logistic Regression на frozen SecBERT embeddings
Той самий протокол, що й B0/B1/B2_S/B2_M: frozen `split_180d`, validation-only вибір
гіперпараметра `C`, **test — рівно один раз**. Основна B3-A використовує ручне зважування
класів `w_pos = n_negative_train / n_positive_train`, узгоджене зі `scale_pos_weight` у B2.
B3-B з `class_weight="balanced"` є окремим sensitivity check і НЕ оцінюється на test.
B3 — sensitivity/upper-bound-style (поточний NVD `description`, не гарантовано доступний на `t_ref`).

Embeddings НЕ перегенеровуються тут — читаються зі Step 4A.9, з'єднуються з таргетами
ЗА `cve_id` (не за порядком рядків — два окремо прочитані parquet-файли можуть,
у принципі, повернути рядки в різному порядку).

Ядро: IDS Research.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

CTI_DIR = Path("/srv/ids_research/data/cti")
RESULTS_DIR = Path.home() / "notebooks" / "2 article" / "release" / "results"

HORIZON = 180
TARGET_COL = f"future_kev_{HORIZON}d"
SPLIT_COL = f"split_{HORIZON}d"
TOP_K_FRACTIONS = [0.001, 0.005, 0.01, 0.05]
RANDOM_STATE = 42
EXPECTED_SPLIT_SIZES = {"train": 249725, "val": 15192, "test": 65159}

C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]   # обирається ЛИШЕ за validation AP, ніколи за test
print(f"HORIZON={HORIZON}, C_GRID={C_GRID}")

HORIZON=180, C_GRID=[0.001, 0.01, 0.1, 1.0, 10.0]


In [2]:
# ---------- Завантаження frozen dataset (таргети/спліт) + frozen embeddings (Step 4A.9) ----------
ds = pd.read_parquet(CTI_DIR / "cve_forecast_dataset.parquet", columns=["cve_id", TARGET_COL, SPLIT_COL])
observed = ds[ds[TARGET_COL].notna()].copy()
observed[TARGET_COL] = observed[TARGET_COL].astype(int)

for split_name, expected in EXPECTED_SPLIT_SIZES.items():
    actual = len(observed[observed[SPLIT_COL] == split_name])
    assert actual == expected, f"{split_name}: actual={actual}, expected={expected} — dataset змінився!"
print("✅ Розміри train/val/test збігаються із зафіксованим Step 4A.2")

embeddings = np.load(RESULTS_DIR / "step4a9_b3_embeddings.npy")
cve_id_order = pd.read_parquet(RESULTS_DIR / "step4a9_b3_cve_id_mapping.parquet")["cve_id"]
assert embeddings.shape[0] == len(cve_id_order)
assert cve_id_order.is_unique
print(f"✅ embeddings: {embeddings.shape}, cve_id_order: {len(cve_id_order):,} унікальних")

# cve_id -> позиція в embeddings (для явного з'єднання, НЕ позиційного припущення)
cve_id_to_row = pd.Series(np.arange(len(cve_id_order)), index=cve_id_order.values)
assert observed["cve_id"].isin(cve_id_to_row.index).all(), \
    "Знайдено cve_id у observed, для яких немає embedding — Step 4A.9 неповний!"
print("✅ Усі observed cve_id мають відповідний embedding")

✅ Розміри train/val/test збігаються із зафіксованим Step 4A.2
✅ embeddings: (379036, 768), cve_id_order: 379,036 унікальних
✅ Усі observed cve_id мають відповідний embedding


In [3]:
# ---------- Побудова X (768-d embeddings) / y для train/val/test, ЗА cve_id ----------
def build_text_matrix(df):
    row_idx = cve_id_to_row.loc[df["cve_id"]].values
    X = embeddings[row_idx]
    assert X.shape[0] == len(df)
    return X

train_df = observed[observed[SPLIT_COL] == "train"].copy()
val_df = observed[observed[SPLIT_COL] == "val"].copy()
test_df = observed[observed[SPLIT_COL] == "test"].copy()

X_train = build_text_matrix(train_df)
X_val = build_text_matrix(val_df)
X_test = build_text_matrix(test_df)
y_train = train_df[TARGET_COL].values
y_val = val_df[TARGET_COL].values
y_test = test_df[TARGET_COL].values

print(f"X_train={X_train.shape}  X_val={X_val.shape}  X_test={X_test.shape}")
assert np.isnan(X_train).sum() == 0 and np.isnan(X_val).sum() == 0 and np.isnan(X_test).sum() == 0
print("✅ NaN-перевірка пройдена для train/val/test embeddings")

X_train=(249725, 768)  X_val=(15192, 768)  X_test=(65159, 768)
✅ NaN-перевірка пройдена для train/val/test embeddings


In [4]:
# ---------- Крок A: вибір C для ОСНОВНОЇ B3-A (manual n_neg/n_pos, узгоджена з B2) ----------
# C обирається ЛИШЕ за validation AP, окремо для manual weighting (НЕ переноситься з balanced —
# абсолютний масштаб ваг взаємодіє з регуляризацією, тож кожен варіант має свій власний C).
n_pos_train = int(y_train.sum())
n_neg_train = len(y_train) - n_pos_train
manual_pos_weight = n_neg_train / n_pos_train
manual_weight = {0: 1.0, 1: float(manual_pos_weight)}

print(f"n_pos_train={n_pos_train}, n_neg_train={n_neg_train}, manual_pos_weight={manual_pos_weight:.6f}")

c_search_results_manual = []
best_c_manual, best_val_ap_manual, best_model_manual = None, -1.0, None

for C in C_GRID:
    clf = LogisticRegression(C=C, class_weight=manual_weight, max_iter=1000,
                             random_state=RANDOM_STATE, solver="lbfgs")
    clf.fit(X_train, y_train)
    val_scores = clf.predict_proba(X_val)[:, 1]
    val_ap = average_precision_score(y_val, val_scores)
    val_roc = roc_auc_score(y_val, val_scores)
    c_search_results_manual.append(dict(C=C, val_ap=val_ap, val_roc_auc=val_roc))
    print(f"  C={C:<8} val_AP={val_ap:.6f}  val_ROC-AUC={val_roc:.4f}")
    if val_ap > best_val_ap_manual:
        best_c_manual, best_val_ap_manual, best_model_manual = C, val_ap, clf

c_search_manual_df = pd.DataFrame(c_search_results_manual)
print(f"\n✅ B3-A: обрано C={best_c_manual} за validation AP={best_val_ap_manual:.6f} (test ще не використовувався)")


n_pos_train=366, n_neg_train=249359, manual_pos_weight=681.308743
  C=0.001    val_AP=0.007847  val_ROC-AUC=0.7898
  C=0.01     val_AP=0.008980  val_ROC-AUC=0.8037
  C=0.1      val_AP=0.009281  val_ROC-AUC=0.7908
  C=1.0      val_AP=0.008496  val_ROC-AUC=0.7671
  C=10.0     val_AP=0.007016  val_ROC-AUC=0.7459

✅ B3-A: обрано C=0.1 за validation AP=0.009281 (test ще не використовувався)


In [5]:
# ---------- Крок B (sensitivity): окремий вибір C для B3-B (sklearn balanced) ----------
# Той самий C_GRID, той самий протокол — але C підбирається ОКРЕМО для balanced weighting,
# а не переноситься з Кроку A. B3-B НІКОЛИ не оцінюється на test (лише sensitivity на validation).
c_search_results_balanced = []
best_c_balanced, best_val_ap_balanced, best_model_balanced = None, -1.0, None

for C in C_GRID:
    clf = LogisticRegression(C=C, class_weight="balanced", max_iter=1000,
                             random_state=RANDOM_STATE, solver="lbfgs")
    clf.fit(X_train, y_train)
    val_scores = clf.predict_proba(X_val)[:, 1]
    val_ap = average_precision_score(y_val, val_scores)
    val_roc = roc_auc_score(y_val, val_scores)
    c_search_results_balanced.append(dict(C=C, val_ap=val_ap, val_roc_auc=val_roc))
    print(f"  C={C:<8} val_AP={val_ap:.6f}  val_ROC-AUC={val_roc:.4f}")
    if val_ap > best_val_ap_balanced:
        best_c_balanced, best_val_ap_balanced, best_model_balanced = C, val_ap, clf

c_search_balanced_df = pd.DataFrame(c_search_results_balanced)
print(f"\n✅ B3-B (sensitivity): обрано C={best_c_balanced} за validation AP={best_val_ap_balanced:.6f}")

# ---- Порівняння НА VALIDATION, кожен зі своїм найкращим C — НЕ обираємо переможця ----
ab_comparison_df = pd.DataFrame([
    dict(variant="B3-A (manual n_neg/n_pos)", best_C=best_c_manual, val_ap=best_val_ap_manual,
        val_roc_auc=c_search_manual_df.loc[c_search_manual_df["C"] == best_c_manual, "val_roc_auc"].iloc[0]),
    dict(variant="B3-B (sklearn balanced)", best_C=best_c_balanced, val_ap=best_val_ap_balanced,
        val_roc_auc=c_search_balanced_df.loc[c_search_balanced_df["C"] == best_c_balanced, "val_roc_auc"].iloc[0]),
])
print("\n" + "=" * 70)
print("Порівняння на VALIDATION (кожен варіант зі СВОЇМ оптимальним C) — sensitivity, не model selection")
print("=" * 70)
display(ab_comparison_df)
print("\nℹ B3-A — ОСНОВНА модель за конвенцією (узгоджена з B2 scale_pos_weight), НЕ обрана за")
print("  цим порівнянням. B3-B ЗАВЖДИ лишається лише sensitivity-довідкою на validation,")
print("  на test НЕ оцінюється взагалі.")

best_model = best_model_manual
best_c = best_c_manual
best_weighting = "manual_n_neg_over_n_pos"


  C=0.001    val_AP=0.008038  val_ROC-AUC=0.7763
  C=0.01     val_AP=0.008575  val_ROC-AUC=0.8037
  C=0.1      val_AP=0.009287  val_ROC-AUC=0.7982
  C=1.0      val_AP=0.008890  val_ROC-AUC=0.7706
  C=10.0     val_AP=0.007173  val_ROC-AUC=0.7506

✅ B3-B (sensitivity): обрано C=0.1 за validation AP=0.009287

Порівняння на VALIDATION (кожен варіант зі СВОЇМ оптимальним C) — sensitivity, не model selection


,variant,best_C,val_ap,val_roc_auc
0,B3-A (manual n_neg/n_pos),0.1,0.009281,0.790829
1,B3-B (sklearn balanced),0.1,0.009287,0.798198



ℹ B3-A — ОСНОВНА модель за конвенцією (узгоджена з B2 scale_pos_weight), НЕ обрана за
  цим порівнянням. B3-B ЗАВЖДИ лишається лише sensitivity-довідкою на validation,
  на test НЕ оцінюється взагалі.


In [6]:
# ---------- Фінальна оцінка на TEST — ОДИН РАЗ, з B3-A (manual n_neg/n_pos) як основною моделлю ----------
# Обрано за КОНВЕНЦІЄЮ (узгодженість з B2 scale_pos_weight), НЕ за validation-показником з
# Кроку B — це навмисне, задокументоване рішення, а не вибір "кращого" з двох варіантів.
test_scores = best_model.predict_proba(X_test)[:, 1]
val_scores_final = best_model.predict_proba(X_val)[:, 1]

ap_test = average_precision_score(y_test, test_scores)
roc_test = roc_auc_score(y_test, test_scores)
ap_val_final = average_precision_score(y_val, val_scores_final)
roc_val_final = roc_auc_score(y_val, val_scores_final)

print(f"B3 (weighting={best_weighting}) val:  AP={ap_val_final:.6f}  ROC-AUC={roc_val_final:.4f}")
print(f"B3 (weighting={best_weighting}) test: AP={ap_test:.6f}  ROC-AUC={roc_test:.4f}  (C={best_c})")

b3b_val_ap_ref = ab_comparison_df.loc[1, "val_ap"]
print(f"Для довідки (sensitivity, лише validation, НЕ test): B3-B (sklearn balanced) val_AP={b3b_val_ap_ref:.6f}")


B3 (weighting=manual_n_neg_over_n_pos) val:  AP=0.009281  ROC-AUC=0.7908
B3 (weighting=manual_n_neg_over_n_pos) test: AP=0.005151  ROC-AUC=0.7760  (C=0.1)
Для довідки (sensitivity, лише validation, НЕ test): B3-B (sklearn balanced) val_AP=0.009287


In [9]:
# ---------- Precision/Recall@K (test) + порівняння з B0/B1/B2_S/B2_M ----------
def precision_recall_at_k(y_true, y_score, k_fraction):
    n = len(y_true)
    k = max(1, int(round(n * k_fraction)))
    order = np.argsort(-y_score, kind="mergesort")   # стабільне сортування, відтворюваність при ties
    top_k_idx = order[:k]
    n_pos_in_top_k = y_true[top_k_idx].sum()
    n_pos_total = y_true.sum()
    return n_pos_in_top_k / k, (n_pos_in_top_k / n_pos_total if n_pos_total else np.nan), k

topk_rows = []
for frac in TOP_K_FRACTIONS:
    p, r, k = precision_recall_at_k(y_test, test_scores, frac)
    topk_rows.append(dict(top_pct=f"{frac:.1%}", K=k, precision=p, recall=r))
topk_b3_df = pd.DataFrame(topk_rows)
print("B3 Precision/Recall@K (test):")
display(topk_b3_df)

b3_row = dict(ap=ap_test, roc_auc=roc_test)
for frac in TOP_K_FRACTIONS:
    row = topk_b3_df[topk_b3_df["top_pct"] == f"{frac:.1%}"].iloc[0]
    b3_row[f"precision@{frac:.1%}"] = row["precision"]
    b3_row[f"recall@{frac:.1%}"] = row["recall"]

frames_to_combine = [pd.DataFrame([b3_row], index=["B3 (sensitivity/upper-bound)"])]
prev_summary_path = RESULTS_DIR / f"step4a4_baseline_summary_h{HORIZON}.csv"
if prev_summary_path.exists():
    frames_to_combine.insert(0, pd.read_csv(prev_summary_path, index_col=0))
b2m_summary_path = RESULTS_DIR / f"step4a5_b2m_summary_h{HORIZON}.csv"
b2m_topk_path = RESULTS_DIR / f"step4a5_b2m_topk_h{HORIZON}.csv"
if b2m_summary_path.exists():
    b2m_prev = pd.read_csv(b2m_summary_path)
    b2m_row = dict(ap=b2m_prev["ap_test"].iloc[0], roc_auc=b2m_prev["roc_auc_test"].iloc[0])
    # Підвантажуємо ВЖЕ зафіксовані test Precision/Recall@K зі Step 4A.5 (той самий формат
    # top_pct/precision/recall, що й topk_b3_df вище) — B2_M НЕ перераховується, лише
    # приєднується правильно до зведеної таблиці, замість того щоб лишити NaN.
    if b2m_topk_path.exists():
        b2m_topk = pd.read_csv(b2m_topk_path)
        for frac in TOP_K_FRACTIONS:
            row = b2m_topk[b2m_topk["top_pct"] == f"{frac:.1%}"]
            if len(row):
                b2m_row[f"precision@{frac:.1%}"] = row["precision"].iloc[0]
                b2m_row[f"recall@{frac:.1%}"] = row["recall"].iloc[0]
    else:
        print(f"⚠ {b2m_topk_path} не знайдено — B2_M Precision/Recall@K залишаться NaN у зведеній таблиці")
    frames_to_combine.insert(1, pd.DataFrame([b2m_row], index=["B2_M (sensitivity/upper-bound)"]))

combined = pd.concat(frames_to_combine)
print("\n" + "=" * 70)
print("ПОРІВНЯННЯ: B0 / B1 / B2_S (frozen) vs B2_M vs B3 (sensitivity/upper-bound)")
print("=" * 70)
display(combined)

B3 Precision/Recall@K (test):


,top_pct,K,precision,recall
0,0.1%,65,0.000000,0.000000
1,0.5%,326,0.009202,0.033333
2,1.0%,652,0.004601,0.033333
3,5.0%,3258,0.006446,0.233333



ПОРІВНЯННЯ: B0 / B1 / B2_S (frozen) vs B2_M vs B3 (sensitivity/upper-bound)


,ap,roc_auc,precision@0.1%,recall@0.1%,precision@0.5%,recall@0.5%,precision@1.0%,recall@1.0%,precision@5.0%,recall@5.0%
B0,0.001381,0.500000,0.001381,0.000998,0.001381,0.005003,0.001381,0.010006,0.001381,0.050001
B1,0.018597,0.715477,0.046154,0.033333,0.042945,0.155556,0.024540,0.177778,0.009515,0.344444
B2_S,0.007880,0.573759,0.061538,0.044444,0.033742,0.122222,0.018405,0.133333,0.005525,0.200000
B2_M (sensitivity/upper-bound),0.011209,0.726713,0.030769,0.022222,0.042945,0.155556,0.027607,0.200000,0.009515,0.344444
B3 (sensitivity/upper-bound),0.005151,0.775979,0.000000,0.000000,0.009202,0.033333,0.004601,0.033333,0.006446,0.233333


In [8]:
# ---------- ОБОВ'ЯЗКОВЕ маркування + збереження ----------
print("=" * 70)
print("⚠ B3 Є SENSITIVITY / UPPER-BOUND-STYLE МОДЕЛЛЮ, НЕ temporally-safe historical forecast.")
print("  description походить із ПОТОЧНОГО знімка NVD і не гарантується як історично доступний")
print("  на t_ref. У Step 4A.3 для 92.07% CVE було виявлено last_modified > t_ref — це")
print("  попередження про potential temporal metadata leakage, а НЕ доказ того, що саме")
print("  description змінився після t_ref.")
print("=" * 70)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
out_row = dict(model="B3_sensitivity_upper_bound", horizon=HORIZON, best_C=best_c_manual,
               ap_test=ap_test, roc_auc_test=roc_test, ap_val=ap_val_final, roc_auc_val=roc_val_final,
               class_weight="manual_n_neg_over_n_pos", manual_pos_weight=manual_pos_weight)
pd.DataFrame([out_row]).to_csv(RESULTS_DIR / f"step4a10_b3_summary_h{HORIZON}.csv", index=False)
c_search_manual_df.to_csv(RESULTS_DIR / f"step4a10_b3_c_search_manual_h{HORIZON}.csv", index=False)
c_search_balanced_df.to_csv(RESULTS_DIR / f"step4a10_b3_c_search_balanced_h{HORIZON}.csv", index=False)
ab_comparison_df.to_csv(RESULTS_DIR / f"step4a10_b3_ab_sensitivity_h{HORIZON}.csv", index=False)
topk_b3_df.to_csv(RESULTS_DIR / f"step4a10_b3_topk_h{HORIZON}.csv", index=False)
print(f"\nЗбережено в {RESULTS_DIR}")

⚠ B3 Є SENSITIVITY / UPPER-BOUND-STYLE МОДЕЛЛЮ, НЕ temporally-safe historical forecast.
  description походить із ПОТОЧНОГО знімка NVD і не гарантується як історично доступний
  на t_ref. У Step 4A.3 для 92.07% CVE було виявлено last_modified > t_ref — це
  попередження про potential temporal metadata leakage, а НЕ доказ того, що саме
  description змінився після t_ref.

Збережено в /home/ivan_r_opirskyy/notebooks/2 article/release/results
